In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time

# Setup Chrome options
chrome_options = Options()
# chrome_options.add_argument('--headless')  # Uncomment to run without GUI
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option('useAutomationExtension', False)

# Initialize driver
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

try:
    print("Opening Waitrose groceries page...")
    driver.get('https://www.waitrose.com/ecom/shop/browse/groceries')
    
    # Wait for page to load
    time.sleep(5)
    
    print(f"Page title: {driver.title}")
    print(f"Current URL: {driver.current_url}")
    
    # Try to find category elements
    categories = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/browse/groceries"]')
    print(f"\nFound {len(categories)} category links")
    
    if len(categories) > 0:
        print("\nFirst 5 categories:")
        for i, cat in enumerate(categories[:5]):
            print(f"  {i+1}. {cat.text} -> {cat.get_attribute('href')}")
    
    # Save page source for inspection
    with open('waitrose_selenium_test.html', 'w', encoding='utf-8') as f:
        f.write(driver.page_source)
    print("\n✓ Saved page source to waitrose_selenium_test.html")
    
except Exception as e:
    print(f"Error: {e}")
    
finally:
    driver.quit()

Opening Waitrose groceries page...
Page title: Shop for your Groceries Online | Waitrose & Partners
Current URL: https://www.waitrose.com/ecom/shop/browse/groceries

Found 344 category links

First 5 categories:
  1.  -> https://www.waitrose.com/ecom/shop/browse/groceries/fresh_and_chilled
  2.  -> https://www.waitrose.com/ecom/shop/browse/groceries/bakery
  3.  -> https://www.waitrose.com/ecom/shop/browse/groceries/food_cupboard
  4.  -> https://www.waitrose.com/ecom/shop/browse/groceries/frozen
  5.  -> https://www.waitrose.com/ecom/shop/browse/groceries/beer_wine_and_spirits

✓ Saved page source to waitrose_selenium_test.html


In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import time

chrome_options = Options()
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

try:
    driver.get('https://www.waitrose.com/ecom/shop/browse/groceries')
    time.sleep(5)
    
    # Find main category containers
    print("=== ANALYZING CATEGORY STRUCTURE ===\n")
    
    # Try different selectors to find categories
    selectors = [
        'div[class*="category"]',
        'li[class*="category"]',
        'a[class*="category"]',
        'div.departments',
        'nav a'
    ]
    
    for selector in selectors:
        elements = driver.find_elements(By.CSS_SELECTOR, selector)
        if elements:
            print(f"✓ Found {len(elements)} elements with selector: {selector}")
            # Print first element's HTML
            if elements:
                print(f"  Sample HTML: {elements[0].get_attribute('outerHTML')[:200]}...\n")
    
    # Look for main categories (top-level)
    print("\n=== MAIN CATEGORIES ===")
    main_cats = driver.find_elements(By.CSS_SELECTOR, 'a[href="/ecom/shop/browse/groceries/"]')
    
    # Try to find unique main categories
    category_urls = set()
    all_links = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/browse/groceries/"]')
    
    for link in all_links:
        href = link.get_attribute('href')
        # Get only first-level categories (one level deep after /groceries/)
        if href and href.count('/') == 7:  # e.g., .../groceries/bakery
            text = link.text.strip()
            if text:
                category_urls.add((text, href))
    
    print(f"\nFound {len(category_urls)} unique main categories:")
    for i, (text, url) in enumerate(sorted(category_urls)[:10], 1):
        print(f"  {i}. {text}: {url}")
    
    # Now let's check a product listing page
    print("\n\n=== CHECKING PRODUCT STRUCTURE ===")
    # Navigate to first category
    if category_urls:
        first_cat = sorted(category_urls)[0][1]
        print(f"\nNavigating to: {first_cat}")
        driver.get(first_cat)
        time.sleep(5)
        
        # Look for product elements
        product_selectors = [
            'div[class*="product"]',
            'article[class*="product"]',
            'li[class*="product"]',
            'div[data-test*="product"]'
        ]
        
        for selector in product_selectors:
            products = driver.find_elements(By.CSS_SELECTOR, selector)
            if products:
                print(f"\n✓ Found {len(products)} products with: {selector}")
                if products:
                    print(f"  Sample: {products[0].get_attribute('outerHTML')[:300]}...")
                break

except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

finally:
    driver.quit()

=== ANALYZING CATEGORY STRUCTURE ===

✓ Found 1 elements with selector: div[class*="category"]
  Sample HTML: <div class="categoryLinks___Fm_gB" data-testid="category-list"><div class="row"><section class="col-xs-12of12"><div class="row"><div class="horizontalDivide___O4ZZX col-xs-12of12 visible-xs-flex visib...

✓ Found 35 elements with selector: nav a
  Sample HTML: <a class="link___kcoPr" href="http://www.waitrosecellar.com/?utm_source=waitrose.com&amp;utm_medium=Referral&amp;utm_campaign=topnav_control_cellar" rel="noopener noreferrer" tabindex="0" target="_bla...


=== MAIN CATEGORIES ===

Found 0 unique main categories:


=== CHECKING PRODUCT STRUCTURE ===


In [3]:
import re
import json

# Read the saved HTML
with open('waitrose_selenium_test.html', 'r', encoding='utf-8') as f:
    html = f.read()

print("=== EXTRACTING CATEGORIES FROM HTML ===\n")

# Find all links with /browse/groceries/ using regex
pattern = r'href="(/ecom/shop/browse/groceries/[^"]*)"'
matches = re.findall(pattern, html)

print(f"Total grocery links found: {len(matches)}\n")

# Extract unique main categories
categories = {}
for href in matches:
    # Parse the category path
    parts = href.split('/browse/groceries/')
    if len(parts) > 1:
        category_path = parts[1].strip('/')
        
        if category_path:  # Not empty
            # Get main category (first segment)
            main_cat = category_path.split('/')[0]
            
            if main_cat and main_cat not in categories:
                categories[main_cat] = {
                    'url': 'https://www.waitrose.com' + href,
                    'full_path': category_path
                }

print(f"Found {len(categories)} unique main categories:\n")
for i, (cat_id, info) in enumerate(sorted(categories.items()), 1):
    name = cat_id.replace('_', ' ').title()
    print(f"{i}. {name}")
    print(f"   URL: {info['url']}\n")

# Save this for reference
with open('categories_found.json', 'w', encoding='utf-8') as f:
    json.dump(categories, f, indent=2)

print(f"✓ Saved {len(categories)} categories to categories_found.json")

=== EXTRACTING CATEGORIES FROM HTML ===

Total grocery links found: 343

Found 34 unique main categories:

1. Baby And Toddler
   URL: https://www.waitrose.com/ecom/shop/browse/groceries/baby_and_toddler

2. Back To School
   URL: https://www.waitrose.com/ecom/shop/browse/groceries/back_to_school/nursery_and_pre-school

3. Bakery
   URL: https://www.waitrose.com/ecom/shop/browse/groceries/bakery

4. Beer Wine And Spirits
   URL: https://www.waitrose.com/ecom/shop/browse/groceries/beer_wine_and_spirits

5. Best Of British
   URL: https://www.waitrose.com/ecom/shop/browse/groceries/best_of_british/fruit_and_vegetables

6. Brandsnew
   URL: https://www.waitrose.com/ecom/shop/browse/groceries/brandsnew

7. Burns Night
   URL: https://www.waitrose.com/ecom/shop/browse/groceries/burns_night/burns_supper_inspiration

8. Christmas
   URL: https://www.waitrose.com/ecom/shop/browse/groceries/christmas/christmas_dinner

9. Dietary And Lifestyle
   URL: https://www.waitrose.com/ecom/shop/browse/gr

In [4]:
import json

# Load categories
with open('categories_found.json', 'r', encoding='utf-8') as f:
    categories = json.load(f)

print("=== ALL CATEGORIES ===\n")
for i, (cat_id, info) in enumerate(sorted(categories.items()), 1):
    name = cat_id.replace('_', ' ').title()
    print(f"{i}. {name}")

print("\n" + "="*50)
print("Which of these are FOOD categories?")
print("="*50)

# Likely food categories (we can refine this)
food_keywords = [
    'bakery', 'fresh', 'frozen', 'meat', 'fish', 'dairy', 'fruit', 'vegetable',
    'food', 'cupboard', 'pantry', 'deli', 'cheese', 'egg', 'ready', 'meal',
    'breakfast', 'lunch', 'dinner', 'snack', 'sweet', 'savoury', 'confection',
    'biscuit', 'cake', 'bread', 'pasta', 'rice', 'cereal', 'condiment', 'sauce',
    'oil', 'spice', 'herb', 'canned', 'jar', 'tin', 'chilled'
]

non_food_keywords = [
    'household', 'health', 'beauty', 'baby', 'toddler', 'pet', 'garden',
    'cleaning', 'laundry', 'toiletries', 'personal_care', 'school'
]

food_categories = {}
non_food_categories = {}

for cat_id, info in categories.items():
    cat_lower = cat_id.lower()
    
    # Check if it's clearly non-food
    if any(keyword in cat_lower for keyword in non_food_keywords):
        non_food_categories[cat_id] = info
    # Check if it's clearly food
    elif any(keyword in cat_lower for keyword in food_keywords):
        food_categories[cat_id] = info
    # Ambiguous - we'll review manually
    else:
        # For now, assume food unless proven otherwise
        food_categories[cat_id] = info

print(f"\n✓ FOOD CATEGORIES ({len(food_categories)}):")
for cat_id in sorted(food_categories.keys()):
    name = cat_id.replace('_', ' ').title()
    print(f"  • {name}")

print(f"\n✗ NON-FOOD CATEGORIES ({len(non_food_categories)}):")
for cat_id in sorted(non_food_categories.keys()):
    name = cat_id.replace('_', ' ').title()
    print(f"  • {name}")

# Save food categories
with open('food_categories.json', 'w', encoding='utf-8') as f:
    json.dump(food_categories, f, indent=2)

print(f"\n✓ Saved {len(food_categories)} food categories to food_categories.json")

=== ALL CATEGORIES ===

1. Baby And Toddler
2. Back To School
3. Bakery
4. Beer Wine And Spirits
5. Best Of British
6. Brandsnew
7. Burns Night
8. Christmas
9. Dietary And Lifestyle
10. Easter
11. Everyday Value
12. First For Welfare
13. Food Cupboard
14. Fresh And Chilled
15. Frozen
16. Health And Wellness
17. Home
18. Household
19. Kitchen Dining And Home
20. Lunar New Year
21. Mothers Day
22. New
23. New Lower Price
24. New Years Eve
25. Newsagents
26. Organic Shop
27. Pancake Day
28. Pet
29. Shop By Occasion
30. Taste Of Japan
31. Tea Coffee And Soft Drinks
32. Toiletries Health And Beauty
33. Valentines Day
34. Waitrose Brands

Which of these are FOOD categories?

✓ FOOD CATEGORIES (28):
  • Bakery
  • Beer Wine And Spirits
  • Best Of British
  • Brandsnew
  • Burns Night
  • Christmas
  • Dietary And Lifestyle
  • Easter
  • Everyday Value
  • First For Welfare
  • Food Cupboard
  • Fresh And Chilled
  • Frozen
  • Home
  • Kitchen Dining And Home
  • Lunar New Year
  • Mothers 

In [5]:
import json

with open('food_categories.json', 'r', encoding='utf-8') as f:
    food_cats = json.load(f)

print(f"=== {len(food_cats)} FOOD CATEGORIES TO SCRAPE ===\n")
for i, cat_id in enumerate(sorted(food_cats.keys()), 1):
    name = cat_id.replace('_', ' ').title()
    print(f"{i:2d}. {name}")

=== 28 FOOD CATEGORIES TO SCRAPE ===

 1. Bakery
 2. Beer Wine And Spirits
 3. Best Of British
 4. Brandsnew
 5. Burns Night
 6. Christmas
 7. Dietary And Lifestyle
 8. Easter
 9. Everyday Value
10. First For Welfare
11. Food Cupboard
12. Fresh And Chilled
13. Frozen
14. Home
15. Kitchen Dining And Home
16. Lunar New Year
17. Mothers Day
18. New
19. New Lower Price
20. New Years Eve
21. Newsagents
22. Organic Shop
23. Pancake Day
24. Shop By Occasion
25. Taste Of Japan
26. Tea Coffee And Soft Drinks
27. Valentines Day
28. Waitrose Brands


start with understanding what data we need to collect from each product.
Step 1: Inspect a Product Page
First, let's navigate to a category and see what product information is available:

In [ ]:
chrome_options = Options()
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

try:
    # Let's check the Bakery category as an example
    print("=== INSPECTING BAKERY CATEGORY ===\n")
    driver.get('https://www.waitrose.com/ecom/shop/browse/groceries/bakery')
    time.sleep(5)
    
    print(f"Page title: {driver.title}")
    print(f"URL: {driver.current_url}\n")
    
    # Look for product containers
    print("=== LOOKING FOR PRODUCTS ===\n")
    
    # Try different selectors
    selectors = [
        '[data-testid="product-tile"]',
        '[data-test="product"]',
        'article',
        'div[class*="product"]',
        'li[class*="product"]'
    ]
    
    products = []
    for selector in selectors:
        elements = driver.find_elements(By.CSS_SELECTOR, selector)
        if elements:
            print(f"✓ Found {len(elements)} products with selector: {selector}")
            products = elements
            break
    
    if products:
        print(f"\n=== ANALYZING FIRST PRODUCT ===\n")
        first_product = products[0]
        
        # Print the HTML structure
        html = first_product.get_attribute('outerHTML')
        print("Product HTML (first 500 chars):")
        print(html[:500])
        print("...\n")
        
        # Try to extract data
        print("=== EXTRACTING DATA ===\n")
        
        # Product name
        name_selectors = ['h2', 'h3', '[class*="name"]', '[data-testid*="name"]']
        for sel in name_selectors:
            try:
                name = first_product.find_element(By.CSS_SELECTOR, sel).text
                if name:
                    print(f"✓ Name: {name}")
                    break
            except:
                pass
        
        # Price
        price_selectors = ['[class*="price"]', '[data-testid*="price"]', 'span[class*="price"]']
        for sel in price_selectors:
            try:
                price = first_product.find_element(By.CSS_SELECTOR, sel).text
                if price:
                    print(f"✓ Price: {price}")
                    break
            except:
                pass
        
        # Image
        try:
            img = first_product.find_element(By.TAG_NAME, 'img')
            print(f"✓ Image URL: {img.get_attribute('src')}")
        except:
            print("✗ No image found")
        
        # Link
        try:
            link = first_product.find_element(By.TAG_NAME, 'a')
            print(f"✓ Product URL: {link.get_attribute('href')}")
        except:
            print("✗ No link found")
        
        # Save full HTML of first product for detailed analysis
        with open('sample_product.html', 'w', encoding='utf-8') as f:
            f.write(html)
        print(f"\n✓ Saved full product HTML to sample_product.html")
        
    else:
        print("✗ No products found. Let's check the page structure...")
        # Save full page HTML
        with open('bakery_page.html', 'w', encoding='utf-8') as f:
            f.write(driver.page_source)
        print("✓ Saved page HTML to bakery_page.html")

except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

finally:
    driver.quit()


=== INSPECTING BAKERY CATEGORY ===

Page title: Bakery | Waitrose & Partners
URL: https://www.waitrose.com/ecom/shop/browse/groceries/bakery

=== LOOKING FOR PRODUCTS ===

✓ Found 50 products with selector: article

=== ANALYZING FIRST PRODUCT ===

Product HTML (first 500 chars):
<article data-testid="product-pod" data-product-availability="available" data-product-id="820327" data-product-name="GAIL's Seeded Sourdough" data-product-type="G" data-product-on-offer="false" data-product-pod-type="sponsored" data-product-index="1" data-product-sponsored="citrusAd-SponsoredProduct-Browse-300119-820327-1" class="visible-xl-flex visible-lg-flex visible-md-flex visible-sm-flex visible-xs-flex productPod___FSa31 col-xl-2of12 col-lg-3of12 col-md-4of12 col-sm-4of12 col-xs-12of12" id
...

=== EXTRACTING DATA ===

✓ Name: GAIL's Seeded Sourdough
✓ Image URL: https://ecom-su-static-prod.wtrecom.com/images/products/3/LN_820327_BP_3.jpg
✓ Product URL: https://www.waitrose.com/ecom/products/gails-seeded

We found products using the article selector. I notice the product has a data-testid="product-pod" attribute and contains useful data attributes. 

In [7]:
chrome_options = Options()
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

try:
    driver.get('https://www.waitrose.com/ecom/shop/browse/groceries/bakery')
    time.sleep(5)
    
    products = driver.find_elements(By.CSS_SELECTOR, 'article[data-testid="product-pod"]')
    
    print(f"=== EXTRACTING DATA FROM FIRST 3 PRODUCTS ===\n")
    
    extracted_products = []
    
    for i, product in enumerate(products[:3], 1):
        print(f"--- PRODUCT {i} ---")
        
        data = {}
        
        # Extract data attributes
        data['product_id'] = product.get_attribute('data-product-id')
        data['name'] = product.get_attribute('data-product-name')
        data['availability'] = product.get_attribute('data-product-availability')
        data['on_offer'] = product.get_attribute('data-product-on-offer')
        data['sponsored'] = product.get_attribute('data-product-sponsored')
        
        # Extract price
        try:
            price_elem = product.find_element(By.CSS_SELECTOR, '[data-testid="product-pod-price"]')
            data['price'] = price_elem.text
        except:
            data['price'] = None
        
        # Extract image
        try:
            img = product.find_element(By.TAG_NAME, 'img')
            data['image_url'] = img.get_attribute('src')
        except:
            data['image_url'] = None
        
        # Extract product URL
        try:
            link = product.find_element(By.CSS_SELECTOR, 'a[href*="/products/"]')
            data['url'] = link.get_attribute('href')
        except:
            data['url'] = None
        
        # Try to get size/weight info
        try:
            size_elem = product.find_element(By.CSS_SELECTOR, '[class*="size"], [class*="weight"]')
            data['size'] = size_elem.text
        except:
            data['size'] = None
        
        extracted_products.append(data)
        
        # Print extracted data
        for key, value in data.items():
            print(f"  {key}: {value}")
        print()
    
    # Save sample data
    with open('sample_products.json', 'w', encoding='utf-8') as f:
        json.dump(extracted_products, f, indent=2)
    
    print("✓ Saved sample products to sample_products.json")
    
    # Check pagination
    print("\n=== CHECKING PAGINATION ===")
    try:
        next_button = driver.find_element(By.CSS_SELECTOR, '[aria-label="Next"]')
        print(f"✓ Found next button: {next_button.get_attribute('outerHTML')[:100]}")
    except:
        print("✗ No next button found")
    
    # Check for "Load More" button
    try:
        load_more = driver.find_element(By.CSS_SELECTOR, 'button[class*="load"], button[class*="more"]')
        print(f"✓ Found load more button: {load_more.text}")
    except:
        print("✗ No load more button found")

except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

finally:
    driver.quit()

print("\n" + "="*60)
print("Now we can define our Item structure based on this data!")
print("="*60)

=== EXTRACTING DATA FROM FIRST 3 PRODUCTS ===

--- PRODUCT 1 ---
  product_id: 820327
  name: GAIL's Seeded Sourdough
  availability: available
  on_offer: false
  sponsored: citrusAd-SponsoredProduct-Browse-300119-820327-1
  price: None
  image_url: https://ecom-su-static-prod.wtrecom.com/images/products/3/LN_820327_BP_3.jpg
  url: https://www.waitrose.com/ecom/products/gails-seeded-sourdough/820327-744449-744450
  size: 650g

--- PRODUCT 2 ---
  product_id: 770979
  name: GAIL's San Francisco Sourdough
  availability: available
  on_offer: false
  sponsored: citrusAd-SponsoredProduct-Browse-300119-770979-2
  price: None
  image_url: https://ecom-su-static-prod.wtrecom.com/images/products/3/LN_770979_BP_3.jpg
  url: https://www.waitrose.com/ecom/products/gails-san-francisco-sourdough/770979-744440-744441
  size: 650g

--- PRODUCT 3 ---
  product_id: 576970
  name: Waitrose Easter 4 Richly Fruited Hot Cross Buns
  availability: available
  on_offer: true
  sponsored: None
  price: None

In [8]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import time

chrome_options = Options()
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

try:
    driver.get('https://www.waitrose.com/ecom/shop/browse/groceries/bakery')
    time.sleep(5)
    
    product = driver.find_elements(By.CSS_SELECTOR, 'article[data-testid="product-pod"]')[0]
    
    print("=== FINDING PRICE ELEMENT ===\n")
    
    # Get the HTML to see price structure
    html = product.get_attribute('innerHTML')
    
    # Look for price patterns in HTML
    import re
    price_patterns = re.findall(r'£\d+\.\d+', html)
    print(f"Found prices in HTML: {price_patterns}\n")
    
    # Try all possible price selectors
    price_selectors = [
        '[data-testid*="price"]',
        '[class*="price"]',
        'span[class*="Price"]',
        'div[class*="price"]',
        'p[class*="price"]',
        '[aria-label*="price"]',
        'span:contains("£")'
    ]
    
    for selector in price_selectors:
        try:
            elements = product.find_elements(By.CSS_SELECTOR, selector)
            if elements:
                print(f"✓ Selector '{selector}' found {len(elements)} elements:")
                for elem in elements[:3]:
                    text = elem.text.strip()
                    if text:
                        print(f"    Text: '{text}'")
                        print(f"    HTML: {elem.get_attribute('outerHTML')[:150]}")
                print()
        except Exception as e:
            pass
    
    # Try to find ANY element with £ symbol
    print("\n=== ALL ELEMENTS CONTAINING '£' ===")
    all_elements = product.find_elements(By.XPATH, ".//*[contains(text(), '£')]")
    for elem in all_elements[:5]:
        print(f"  Text: '{elem.text}'")
        print(f"  Tag: {elem.tag_name}, Class: {elem.get_attribute('class')}")
        print()

except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

finally:
    driver.quit()

=== FINDING PRICE ELEMENT ===

Found prices in HTML: ['£4.95', '£7.62']

✓ Selector '[class*="price"]' found 2 elements:

✓ Selector 'span[class*="Price"]' found 1 elements:

✓ Selector 'div[class*="price"]' found 1 elements:


=== ALL ELEMENTS CONTAINING '£' ===
  Text: ''
  Tag: span, Class: 

  Text: ''
  Tag: span, Class: pricePerUnit___a1PxI priceInfo___ThE1M



In [9]:
chrome_options = Options()
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

try:
    driver.get('https://www.waitrose.com/ecom/shop/browse/groceries/bakery')
    time.sleep(5)
    
    product = driver.find_elements(By.CSS_SELECTOR, 'article[data-testid="product-pod"]')[0]
    
    print("=== CHECKING PRICE STRUCTURE ===\n")
    
    # Get all span elements with price classes
    price_spans = product.find_elements(By.CSS_SELECTOR, 'span[class*="rice"]')
    
    for i, span in enumerate(price_spans, 1):
        print(f"--- Span {i} ---")
        print(f"Text: '{span.text}'")
        print(f"innerHTML: {span.get_attribute('innerHTML')}")
        print(f"innerText: {span.get_attribute('innerText')}")
        print(f"textContent: {span.get_attribute('textContent')}")
        print(f"Class: {span.get_attribute('class')}")
        print(f"Full HTML: {span.get_attribute('outerHTML')[:200]}")
        print()
    
    # Try a different approach - get the entire price container
    print("\n=== LOOKING FOR PRICE CONTAINER ===")
    price_containers = product.find_elements(By.CSS_SELECTOR, 'div[class*="rice"], footer')
    
    for container in price_containers[:3]:
        print(f"Container text: '{container.text}'")
        print(f"Container HTML: {container.get_attribute('outerHTML')[:300]}")
        print()
    
    # Try to extract using JavaScript
    print("\n=== USING JAVASCRIPT TO EXTRACT ===")
    price_js = driver.execute_script("""
        const product = arguments[0];
        const priceElements = product.querySelectorAll('[class*="rice"]');
        return Array.from(priceElements).map(el => ({
            text: el.textContent,
            innerHTML: el.innerHTML,
            className: el.className
        }));
    """, product)
    
    print("JavaScript extraction:")
    for item in price_js:
        print(f"  Text: '{item['text']}'")
        print(f"  Class: {item['className']}")
        print()

except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

finally:
    driver.quit()

=== CHECKING PRICE STRUCTURE ===

--- Span 1 ---
Text: ''
innerHTML: <p class="sr-only">Item price</p><span class="">£4.95</span>
innerText: Item price

£4.95
textContent: Item price£4.95
Class: itemPrice___j1MYI
Full HTML: <span data-test="product-pod-price" class="itemPrice___j1MYI"><p class="sr-only">Item price</p><span class="">£4.95</span></span>

--- Span 2 ---
Text: ''
innerHTML: <p class="sr-only">Price per unit</p>£7.62/kg
innerText: Price per unit

£7.62/kg
textContent: Price per unit£7.62/kg
Class: pricePerUnit___a1PxI priceInfo___ThE1M
Full HTML: <span class="pricePerUnit___a1PxI priceInfo___ThE1M"><p class="sr-only">Price per unit</p>£7.62/kg</span>


=== LOOKING FOR PRICE CONTAINER ===
Container text: ''
Container HTML: <div class="prices___IA5LC"><span data-test="product-pod-price" class="itemPrice___j1MYI"><p class="sr-only">Item price</p><span class="">£4.95</span></span><span class="pricePerUnit___a1PxI priceInfo___ThE1M"><p class="sr-only">Price per unit</p>£7.62/kg<

In [10]:
import re

chrome_options = Options()
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

try:
    driver.get('https://www.waitrose.com/ecom/shop/browse/groceries/bakery')
    time.sleep(5)
    
    products = driver.find_elements(By.CSS_SELECTOR, 'article[data-testid="product-pod"]')
    
    print(f"=== EXTRACTING COMPLETE DATA FROM 3 PRODUCTS ===\n")
    
    for i, product in enumerate(products[:3], 1):
        print(f"--- PRODUCT {i} ---")
        
        # Basic attributes
        product_id = product.get_attribute('data-product-id')
        name = product.get_attribute('data-product-name')
        
        # Price
        price = None
        price_per_unit = None
        try:
            price_elem = product.find_element(By.CSS_SELECTOR, '[data-test="product-pod-price"]')
            price_text = price_elem.get_attribute('textContent')
            # Extract £X.XX pattern
            price_match = re.search(r'£(\d+\.\d+)', price_text)
            if price_match:
                price = price_match.group(0)
        except:
            pass
        
        # Price per unit
        try:
            unit_elem = product.find_element(By.CSS_SELECTOR, '.pricePerUnit___a1PxI')
            unit_text = unit_elem.get_attribute('textContent')
            # Extract £X.XX/unit pattern
            unit_match = re.search(r'£[\d.]+/\w+', unit_text)
            if unit_match:
                price_per_unit = unit_match.group(0)
        except:
            pass
        
        # Size
        size = None
        try:
            # Size is often in the product name or separate element
            size_elem = product.find_element(By.CSS_SELECTOR, '[class*="size"]')
            size = size_elem.get_attribute('textContent').strip()
        except:
            # Try to find it in data attributes or text
            try:
                size_divs = product.find_elements(By.TAG_NAME, 'div')
                for div in size_divs:
                    text = div.get_attribute('textContent')
                    if re.search(r'\d+\s*(g|kg|ml|l|pack)', text, re.IGNORECASE):
                        size = text.strip()
                        break
            except:
                pass
        
        # Image & URL
        image_url = None
        product_url = None
        try:
            img = product.find_element(By.TAG_NAME, 'img')
            image_url = img.get_attribute('src')
        except:
            pass
        
        try:
            link = product.find_element(By.CSS_SELECTOR, 'a[href*="/products/"]')
            product_url = link.get_attribute('href')
        except:
            pass
        
        print(f"  ID: {product_id}")
        print(f"  Name: {name}")
        print(f"  Price: {price}")
        print(f"  Price per unit: {price_per_unit}")
        print(f"  Size: {size}")
        print(f"  Image: {image_url}")
        print(f"  URL: {product_url}")
        print()

except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

finally:
    driver.quit()

print("="*60)
print("✓ Data extraction method verified!")
print("Now we can build the complete scraper structure.")
print("="*60)

=== EXTRACTING COMPLETE DATA FROM 3 PRODUCTS ===

--- PRODUCT 1 ---
  ID: 820327
  Name: GAIL's Seeded Sourdough
  Price: £4.95
  Price per unit: £7.62/kg
  Size: 650g
  Image: https://ecom-su-static-prod.wtrecom.com/images/products/3/LN_820327_BP_3.jpg
  URL: https://www.waitrose.com/ecom/products/gails-seeded-sourdough/820327-744449-744450

--- PRODUCT 2 ---
  ID: 770979
  Name: GAIL's San Francisco Sourdough
  Price: £4.95
  Price per unit: £7.62/kg
  Size: 650g
  Image: https://ecom-su-static-prod.wtrecom.com/images/products/3/LN_770979_BP_3.jpg
  URL: https://www.waitrose.com/ecom/products/gails-san-francisco-sourdough/770979-744440-744441

--- PRODUCT 3 ---
  ID: 576970
  Name: Waitrose Easter 4 Richly Fruited Hot Cross Buns
  Price: £2.00
  Price per unit: None
  Size: 4s
  Image: https://ecom-su-static-prod.wtrecom.com/images/products/3/LN_576970_BP_3.jpg
  URL: https://www.waitrose.com/ecom/products/waitrose-easter-4-richly-fruited-hot-cross-buns/576970-794942-794943

✓ Data e

In [11]:
import json

# Load all categories
with open('categories_found.json', 'r') as f:
    all_categories = json.load(f)

# Define non-food keywords
non_food_keywords = [
    'household', 'health', 'beauty', 'baby', 'toddler', 'pet', 
    'toiletries', 'personal_care', 'school', 'home', 'kitchen_dining'
]

# Filter to only food categories
food_categories = {}
for cat_id, info in all_categories.items():
    cat_lower = cat_id.lower()
    
    # Exclude non-food categories
    if not any(keyword in cat_lower for keyword in non_food_keywords):
        food_categories[cat_id] = info

print(f"Created {len(food_categories)} food categories from {len(all_categories)} total categories")

# Save as food_categories.json
with open('food_categories.json', 'w', encoding='utf-8') as f:
    json.dump(food_categories, f, indent=2)

print("✓ Saved to food_categories.json")

# Show the categories
for i, cat_id in enumerate(sorted(food_categories.keys()), 1):
    name = cat_id.replace('_', ' ').title()
    print(f"{i:2d}. {name}")

Created 26 food categories from 34 total categories
✓ Saved to food_categories.json
 1. Bakery
 2. Beer Wine And Spirits
 3. Best Of British
 4. Brandsnew
 5. Burns Night
 6. Christmas
 7. Dietary And Lifestyle
 8. Easter
 9. Everyday Value
10. First For Welfare
11. Food Cupboard
12. Fresh And Chilled
13. Frozen
14. Lunar New Year
15. Mothers Day
16. New
17. New Lower Price
18. New Years Eve
19. Newsagents
20. Organic Shop
21. Pancake Day
22. Shop By Occasion
23. Taste Of Japan
24. Tea Coffee And Soft Drinks
25. Valentines Day
26. Waitrose Brands


Data extraction works perfectly. Now let's build the actual scraper structure.
Step 2: Create the Scrapy Project Structure
First, let's set up the Scrapy project inside scraper/:

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import time
import json
import re

# Setup Chrome with stealth options
chrome_options = Options()
# DON'T run headless - show the browser
# chrome_options.add_argument('--headless')  # COMMENTED OUT

chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option('useAutomationExtension', False)
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--no-sandbox')

# Add a more realistic user agent
chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

# Additional stealth: execute CDP commands
driver.execute_cdp_cmd('Network.setUserAgentOverride', {
    "userAgent": 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
})
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

print("Testing with visible browser (non-headless)...")

try:
    url = "https://www.waitrose.com/ecom/shop/browse/groceries/bakery"
    print(f"\nNavigating to: {url}")
    
    driver.get(url)
    print("Waiting 8 seconds for page to load...")
    time.sleep(8)
    
    print(f"Page title: {driver.title}")
    
    # Find products
    products = driver.find_elements(By.CSS_SELECTOR, 'article[data-testid="product-pod"]')
    print(f"\nFound {len(products)} products")
    
    if len(products) > 0:
        print("\n✓ SUCCESS! Products found with non-headless browser")
        print("\nExtracting first 3 products...")
        
        for i, product in enumerate(products[:3], 1):
            product_id = product.get_attribute('data-product-id')
            name = product.get_attribute('data-product-name')
            print(f"{i}. {name} (ID: {product_id})")
        
        print("\n✓ Non-headless mode works!")
    else:
        print("\n✗ Still no products. Checking page...")
        with open('test_page_nonheadless.html', 'w', encoding='utf-8') as f:
            f.write(driver.page_source)

except Exception as e:
    print(f"\n✗ Error: {e}")
    import traceback
    traceback.print_exc()

finally:
    input("\nPress Enter to close browser...")
    driver.quit()

Testing with visible browser (non-headless)...

Navigating to: https://www.waitrose.com/ecom/shop/browse/groceries/bakery
Waiting 8 seconds for page to load...
Page title: Bakery | Waitrose & Partners

Found 50 products

✓ SUCCESS! Products found with non-headless browser

Extracting first 3 products...
1. GAIL's Seeded Sourdough (ID: 820327)
2. Waitrose Easter 4 Richly Fruited Hot Cross Buns (ID: 576970)
3. GAIL's San Francisco Sourdough (ID: 770979)

✓ Non-headless mode works!
